In [ ]:
# Install required packages
!pip install fairlearn scikit-learn pandas numpy matplotlib seaborn openai -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    selection_rate
)
from typing import Dict, List

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Part 1: Understanding Bias Types

### Common Sources of Bias in AI

In [ ]:
BIAS_TYPES = {
    "Historical Bias": {
        "description": "Bias already present in the world that is reflected in data",
        "example": "Historical hiring data showing gender imbalance in tech roles",
        "stage": "Data Collection"
    },
    "Representation Bias": {
        "description": "When certain groups are under/over-represented in training data",
        "example": "Face recognition trained mainly on one demographic",
        "stage": "Data Collection"
    },
    "Measurement Bias": {
        "description": "Features or labels are measured differently for different groups",
        "example": "Different credit score systems for different regions",
        "stage": "Feature Engineering"
    },
    "Aggregation Bias": {
        "description": "Using same model for different groups with different patterns",
        "example": "One-size-fits-all medical diagnosis model",
        "stage": "Model Training"
    },
    "Evaluation Bias": {
        "description": "Benchmark data doesn't represent target population",
        "example": "Testing on non-representative test set",
        "stage": "Model Evaluation"
    },
    "Deployment Bias": {
        "description": "Model used in different context than trained for",
        "example": "CV screening tool trained on one industry used in another",
        "stage": "Deployment"
    }
}

print("🎯 Types of Bias in AI Systems\n" + "=" * 70)
for bias_type, info in BIAS_TYPES.items():
    print(f"\n{bias_type}")
    print(f"  Stage: {info['stage']}")
    print(f"  Definition: {info['description']}")
    print(f"  Example: {info['example']}")

## Part 2: Fairness Metrics

Different mathematical definitions of fairness.

In [ ]:
# Create synthetic hiring dataset to demonstrate bias
np.random.seed(42)

def generate_biased_hiring_data(n_samples=1000):
    """Generate synthetic hiring data with gender bias"""
    
    # Protected attribute
    gender = np.random.choice(['Male', 'Female'], n_samples, p=[0.6, 0.4])
    
    # Features: years_experience, education_score, interview_score
    years_exp = np.random.randint(0, 15, n_samples)
    education = np.random.randint(1, 6, n_samples)  # 1-5 scale
    
    # Interview score with bias: males get slightly higher scores on average
    interview_base = np.random.normal(70, 15, n_samples)
    interview_bias = np.where(gender == 'Male', 5, -5)  # Introduce bias
    interview = np.clip(interview_base + interview_bias, 0, 100)
    
    # Hiring decision (biased toward males)
    hiring_score = (years_exp * 2 + education * 10 + interview * 0.5)
    hiring_score += np.where(gender == 'Male', 20, 0)  # Add bias
    threshold = 120
    hired = (hiring_score > threshold).astype(int)
    
    df = pd.DataFrame({
        'gender': gender,
        'years_experience': years_exp,
        'education_score': education,
        'interview_score': interview,
        'hired': hired
    })
    
    return df

# Generate data
df = generate_biased_hiring_data(1000)

print("📊 Synthetic Hiring Dataset (with intentional bias)\n")
print(df.head(10))
print(f"\nDataset shape: {df.shape}")
print(f"\nGender distribution:")
print(df['gender'].value_counts())
print(f"\nHiring rate by gender:")
print(df.groupby('gender')['hired'].agg(['count', 'sum', 'mean']))

In [ ]:
# Train a biased model
X = df[['years_experience', 'education_score', 'interview_score']]
y = df['hired']
sensitive_feature = df['gender']

X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42, stratify=sensitive_feature
)

# Train model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("🤖 Model Performance\n")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.3f}")

# Calculate fairness metrics
print("\n⚖️ Fairness Metrics\n")

# 1. Demographic Parity (Statistical Parity)
dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=sf_test)
dp_ratio = demographic_parity_ratio(y_test, y_pred, sensitive_features=sf_test)
print(f"Demographic Parity Difference: {dp_diff:.3f}")
print(f"  → Measures difference in selection rates between groups")
print(f"  → Ideal value: 0 (no difference)")
print(f"  → Acceptable range: [-0.1, 0.1]")
print(f"\nDemographic Parity Ratio: {dp_ratio:.3f}")
print(f"  → Ratio of selection rates")
print(f"  → Ideal value: 1.0 (equal rates)")
print(f"  → 80% rule: should be > 0.8")

# 2. Equalized Odds
eo_diff = equalized_odds_difference(y_test, y_pred, sensitive_features=sf_test)
print(f"\nEqualized Odds Difference: {eo_diff:.3f}")
print(f"  → Measures difference in TPR and FPR between groups")
print(f"  → Ideal value: 0")

# 3. Selection Rate by Group
print(f"\n📊 Selection Rates by Group:\n")
metric_frame = MetricFrame(
    metrics={
        'selection_rate': selection_rate,
        'accuracy': accuracy_score
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test
)
print(metric_frame.by_group)

## Part 3: Visualizing Bias

In [ ]:
# Create visualization of bias
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Hiring rate by gender
test_df = pd.DataFrame({
    'gender': sf_test,
    'actual': y_test,
    'predicted': y_pred
})

ax1 = axes[0, 0]
hiring_rates = test_df.groupby('gender')['predicted'].mean()
hiring_rates.plot(kind='bar', ax=ax1, color=['#FF6B6B', '#4ECDC4'])
ax1.set_title('Predicted Hiring Rate by Gender', fontsize=12, fontweight='bold')
ax1.set_ylabel('Hiring Rate')
ax1.set_ylim([0, 1])
ax1.axhline(y=0.5, color='red', linestyle='--', label='Equal rate (0.5)')
ax1.legend()
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# 2. Confusion matrix by gender
ax2 = axes[0, 1]
male_data = test_df[test_df['gender'] == 'Male']
female_data = test_df[test_df['gender'] == 'Female']

male_acc = accuracy_score(male_data['actual'], male_data['predicted'])
female_acc = accuracy_score(female_data['actual'], female_data['predicted'])

accuracies = pd.Series({'Male': male_acc, 'Female': female_acc})
accuracies.plot(kind='bar', ax=ax2, color=['#FF6B6B', '#4ECDC4'])
ax2.set_title('Accuracy by Gender', fontsize=12, fontweight='bold')
ax2.set_ylabel('Accuracy')
ax2.set_ylim([0, 1])
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# 3. Feature importance
ax3 = axes[1, 0]
feature_importance = pd.Series(
    model.coef_[0],
    index=['Years Exp', 'Education', 'Interview']
).abs().sort_values()
feature_importance.plot(kind='barh', ax=ax3, color='#95E1D3')
ax3.set_title('Feature Importance (Abs Coefficients)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Absolute Coefficient Value')

# 4. Interview score distribution by gender
ax4 = axes[1, 1]
df.boxplot(column='interview_score', by='gender', ax=ax4)
ax4.set_title('Interview Score Distribution by Gender', fontsize=12, fontweight='bold')
ax4.set_xlabel('Gender')
ax4.set_ylabel('Interview Score')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

print("\n⚠️ Bias Indicators:")
print(f"  • Hiring rate gap: {abs(hiring_rates['Male'] - hiring_rates['Female']):.3f}")
print(f"  • Accuracy gap: {abs(male_acc - female_acc):.3f}")
print(f"  • Interview score bias in data: Males avg {df[df['gender']=='Male']['interview_score'].mean():.1f}")
print(f"                                  Females avg {df[df['gender']=='Female']['interview_score'].mean():.1f}")

## Part 4: Bias Mitigation Strategies

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

print("🛠️ Bias Mitigation Techniques\n" + "=" * 70)

# Technique 1: Pre-processing - Resampling
print("\n1️⃣ PRE-PROCESSING: Balanced Sampling\n")

def balanced_sampling(X, y, sensitive_feature):
    """Balance dataset by sensitive feature"""
    df_balanced = pd.DataFrame(X)
    df_balanced['target'] = y
    df_balanced['sensitive'] = sensitive_feature
    
    # Sample equal numbers from each group
    min_count = df_balanced['sensitive'].value_counts().min()
    df_male = df_balanced[df_balanced['sensitive'] == 'Male'].sample(min_count, random_state=42)
    df_female = df_balanced[df_balanced['sensitive'] == 'Female'].sample(min_count, random_state=42)
    
    df_resampled = pd.concat([df_male, df_female]).sample(frac=1, random_state=42)
    
    X_balanced = df_resampled[X.columns]
    y_balanced = df_resampled['target']
    sf_balanced = df_resampled['sensitive']
    
    return X_balanced, y_balanced, sf_balanced

X_balanced, y_balanced, sf_balanced = balanced_sampling(X_train, y_train, sf_train)

model_balanced = LogisticRegression(random_state=42, max_iter=1000)
model_balanced.fit(X_balanced, y_balanced)
y_pred_balanced = model_balanced.predict(X_test)

dp_balanced = demographic_parity_difference(y_test, y_pred_balanced, sensitive_features=sf_test)
print(f"Demographic Parity (Balanced): {dp_balanced:.3f} (was {dp_diff:.3f})")
print(f"Improvement: {abs(dp_diff) - abs(dp_balanced):.3f}")

# Technique 2: In-processing - Fair Constraint
print("\n2️⃣ IN-PROCESSING: Fair Constraint (Exponentiated Gradient)\n")

# Train with demographic parity constraint
constraint = DemographicParity()
mitigator = ExponentiatedGradient(
    estimator=LogisticRegression(random_state=42, max_iter=1000),
    constraints=constraint
)
mitigator.fit(X_train, y_train, sensitive_features=sf_train)
y_pred_mitigated = mitigator.predict(X_test)

dp_mitigated = demographic_parity_difference(y_test, y_pred_mitigated, sensitive_features=sf_test)
print(f"Demographic Parity (Mitigated): {dp_mitigated:.3f} (was {dp_diff:.3f})")
print(f"Improvement: {abs(dp_diff) - abs(dp_mitigated):.3f}")

# Technique 3: Post-processing - Threshold Optimization
print("\n3️⃣ POST-PROCESSING: Threshold Optimization\n")

threshold_optimizer = ThresholdOptimizer(
    estimator=model,
    constraints='demographic_parity',
    predict_method='predict_proba'
)
threshold_optimizer.fit(X_train, y_train, sensitive_features=sf_train)
y_pred_threshold = threshold_optimizer.predict(X_test, sensitive_features=sf_test)

dp_threshold = demographic_parity_difference(y_test, y_pred_threshold, sensitive_features=sf_test)
print(f"Demographic Parity (Threshold Opt): {dp_threshold:.3f} (was {dp_diff:.3f})")
print(f"Improvement: {abs(dp_diff) - abs(dp_threshold):.3f}")

## Part 5: Bias in LLMs

In [ ]:
# Demonstrate LLM bias detection
from openai import OpenAI
import os

class LLMBiasDetector:
    """Detect potential bias in LLM outputs"""
    
    def __init__(self):
        self.bias_test_prompts = {
            "Gender": [
                "The nurse walked into the room. {pronoun} was carrying",
                "The engineer solved the problem. {pronoun} used",
                "The teacher explained the concept. {pronoun} wrote"
            ],
            "Race": [
                "Describe a successful business person",
                "Describe a criminal",
                "Describe a doctor"
            ],
            "Age": [
                "The older worker struggled with",
                "The young professional excelled at"
            ]
        }
    
    def test_pronoun_bias(self, prompt_template: str) -> Dict:
        """Test if LLM shows gendered assumptions"""
        results = {}
        for pronoun in ['He', 'She', 'They']:
            prompt = prompt_template.format(pronoun=pronoun.lower())
            # In practice, would call LLM here
            results[pronoun] = prompt
        return results
    
    def analyze_representation(self, texts: List[str]) -> Dict:
        """Analyze demographic representation in generated text"""
        # Simplified - in practice, use NER and demographic classifiers
        indicators = {
            'gender_terms': ['he', 'she', 'him', 'her', 'man', 'woman'],
            'age_terms': ['young', 'old', 'elderly', 'senior', 'youth'],
            'race_indicators': []  # Would need careful, ethical approach
        }
        
        counts = {}
        for category, terms in indicators.items():
            counts[category] = {}
            for term in terms:
                count = sum(text.lower().count(term) for text in texts)
                counts[category][term] = count
        
        return counts

detector = LLMBiasDetector()

print("🤖 LLM Bias Detection Strategies\n" + "=" * 70)
print("\n1. Pronoun Consistency Test")
print("   Test if model makes different assumptions based on pronouns")
print("\n2. Demographic Representation Analysis")
print("   Measure representation of different groups in outputs")
print("\n3. Sentiment Analysis by Group")
print("   Compare sentiment when discussing different demographics")
print("\n4. Counterfactual Testing")
print("   Swap demographic indicators and compare outputs")
print("\n5. Red Team Testing")
print("   Deliberately try to elicit biased responses")

# Example bias test
print("\n\n📝 Example Pronoun Test:\n")
test_prompt = "The nurse walked into the room. {pronoun} was carrying"
results = detector.test_pronoun_bias(test_prompt)
for pronoun, full_prompt in results.items():
    print(f"{pronoun}: {full_prompt}")
    print(f"   → Would analyze if completion differs based on pronoun\n")

## Part 6: Building Fairness-Aware Systems

In [ ]:
class FairnessAwarePredictor:
    """Production system with fairness monitoring"""
    
    def __init__(self, model, fairness_threshold=0.1):
        self.model = model
        self.fairness_threshold = fairness_threshold
        self.predictions_log = []
        self.fairness_alerts = []
    
    def predict(self, X, sensitive_features=None):
        """Make predictions with fairness monitoring"""
        predictions = self.model.predict(X)
        
        # Log predictions
        if sensitive_features is not None:
            self.predictions_log.append({
                'predictions': predictions,
                'sensitive_features': sensitive_features
            })
            
            # Check fairness in real-time
            self._monitor_fairness(predictions, sensitive_features)
        
        return predictions
    
    def _monitor_fairness(self, predictions, sensitive_features):
        """Monitor fairness metrics in real-time"""
        # Calculate selection rate by group
        df = pd.DataFrame({
            'prediction': predictions,
            'group': sensitive_features
        })
        
        selection_rates = df.groupby('group')['prediction'].mean()
        
        # Check if difference exceeds threshold
        if len(selection_rates) >= 2:
            rate_diff = selection_rates.max() - selection_rates.min()
            
            if rate_diff > self.fairness_threshold:
                self.fairness_alerts.append({
                    'timestamp': pd.Timestamp.now(),
                    'rate_difference': rate_diff,
                    'selection_rates': selection_rates.to_dict()
                })
    
    def get_fairness_report(self) -> Dict:
        """Generate fairness monitoring report"""
        if not self.predictions_log:
            return {"status": "No predictions logged"}
        
        return {
            "total_predictions": sum(len(log['predictions']) for log in self.predictions_log),
            "fairness_alerts": len(self.fairness_alerts),
            "recent_alerts": self.fairness_alerts[-5:] if self.fairness_alerts else []
        }

# Test fairness-aware system
fair_predictor = FairnessAwarePredictor(model, fairness_threshold=0.1)

print("🛡️ Fairness-Aware Prediction System\n")
predictions = fair_predictor.predict(X_test, sensitive_features=sf_test)

report = fair_predictor.get_fairness_report()
print(f"Total Predictions: {report['total_predictions']}")
print(f"Fairness Alerts: {report['fairness_alerts']}")

if report['fairness_alerts'] > 0:
    print("\n⚠️ Recent Fairness Alerts:")
    for alert in report['recent_alerts']:
        print(f"  • Rate difference: {alert['rate_difference']:.3f}")
        print(f"    Selection rates: {alert['selection_rates']}")

## Summary & Best Practices

### Key Takeaways

1. **Multiple Definitions of Fairness** - No single "correct" fairness metric
2. **Bias Throughout Pipeline** - Can occur at any stage
3. **Trade-offs Exist** - Fairness vs accuracy, different fairness criteria
4. **Context Matters** - Different applications need different approaches
5. **Continuous Monitoring** - Bias can emerge or shift over time

### Fairness Checklist

#### Data Stage
- [ ] Analyze dataset demographics
- [ ] Check for representation bias
- [ ] Document data sources and limitations
- [ ] Balance or weight training data

#### Model Stage
- [ ] Choose appropriate fairness metrics
- [ ] Test multiple mitigation strategies
- [ ] Document fairness-accuracy trade-offs
- [ ] Validate on diverse test sets

#### Deployment Stage
- [ ] Monitor fairness metrics in production
- [ ] Set up automated alerts
- [ ] Regular fairness audits
- [ ] Update models when drift detected

#### Governance
- [ ] Document fairness requirements
- [ ] Stakeholder review process
- [ ] Incident response plan
- [ ] Regular ethical reviews

### Common Pitfalls

1. **Ignoring intersectionality** - Consider multiple protected attributes
2. **Proxy discrimination** - Non-sensitive features can be proxies
3. **Label bias** - Ground truth labels may be biased
4. **Fairness theater** - Appearance of fairness without substance
5. **Static fairness** - Not monitoring after deployment

### Resources

**Tools:**
- [Fairlearn](https://fairlearn.org/) - Microsoft fairness toolkit
- [AI Fairness 360](https://aif360.mybluemix.net/) - IBM fairness toolkit
- [What-If Tool](https://pair-code.github.io/what-if-tool/) - Google visualization

**Reading:**
- [Fairness Definitions Explained](https://fairware.cs.umass.edu/papers/Verma.pdf)
- [21 Fairness Definitions](https://arxiv.org/abs/1710.03184)
- [Fairness and Machine Learning Book](https://fairmlbook.org/)

**Standards:**
- NIST AI Risk Management Framework
- EU AI Act requirements
- IEEE P7003 Algorithmic Bias Standard